In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# 1 Load and prepare dataset
df = pd.read_csv("gestures.csv")

X = df.drop("label", axis=1).values
y = df["label"].values

num_classes = len(np.unique(y))
print(f"Loaded {len(df)} rows across {num_classes} gesture classes.")

Loaded 2503 rows across 6 gesture classes.


In [3]:
# 2. Train / Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize feature vectors
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

In [4]:
# 3. Define Lightweight 3-Layer ANN
class GestureANN(nn.Module):
    def __init__(self, input_dim=42, num_classes=4):
        super(GestureANN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = GestureANN(input_dim=42, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
# 4. Train Model
epochs = 60
print("\n--- Training Model ---")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.4f}")


--- Training Model ---
Epoch [10/60] - Loss: 1.6227
Epoch [20/60] - Loss: 1.4119
Epoch [30/60] - Loss: 1.1439
Epoch [40/60] - Loss: 0.8563
Epoch [50/60] - Loss: 0.6097
Epoch [60/60] - Loss: 0.4221


In [6]:
# 5. Evaluate Accuracy
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, predicted = torch.max(test_outputs, 1)
    correct = (predicted == y_test_t).sum().item()
    accuracy = (correct / len(y_test_t)) * 100
    print(f"\n✅ Final Test Accuracy: {accuracy:.2f}%")

# 6. Save Model Checkpoint
torch.save({'model_state': model.state_dict(), 'scaler': scaler, 'num_classes': num_classes}, "gesture_ann_model.pth")
print("💾 Model saved successfully as 'gesture_ann_model.pth'!")


✅ Final Test Accuracy: 96.81%
💾 Model saved successfully as 'gesture_ann_model.pth'!
